In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class UnderwaterSegDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, mask_transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        mask = remap_mask(mask)

        return image, mask

print(f"Dataset contents: {os.listdir(path)}")

train_image_dir = None
train_mask_dir = None
test_image_dir = None
test_mask_dir = None

if os.path.exists(os.path.join(path, "train")):
    if os.path.exists(os.path.join(path, "train", "images")):
        train_image_dir = os.path.join(path, "train", "images")
        train_mask_dir = os.path.join(path, "train", "masks")
        test_image_dir = os.path.join(path, "test", "images")
        test_mask_dir = os.path.join(path, "test", "masks")
    else:
        train_image_dir = os.path.join(path, "train")
        train_mask_dir = os.path.join(path, "train")
        test_image_dir = os.path.join(path, "test")
        test_mask_dir = os.path.join(path, "test")
elif os.path.exists(os.path.join(path, "images")):
    train_image_dir = os.path.join(path, "images")
    train_mask_dir = os.path.join(path, "masks")
    test_image_dir = train_image_dir
    test_mask_dir = train_mask_dir
else:
    for item in os.listdir(path):
        item_path = os.path.join(path, item)
        if os.path.isdir(item_path):
            if os.path.exists(os.path.join(item_path, "images")):
                train_image_dir = os.path.join(item_path, "images")
                train_mask_dir = os.path.join(item_path, "masks")
                test_image_dir = train_image_dir
                test_mask_dir = train_mask_dir
                break

if train_image_dir is None:
    raise FileNotFoundError(f"Could not find images in {path}. Please check dataset structure.")

print(f"Using train_image_dir: {train_image_dir}")
print(f"Using train_mask_dir: {train_mask_dir}")

train_images = sorted(glob.glob(os.path.join(train_image_dir, "*.jpg")))
if len(train_images) == 0:
    train_images = sorted(glob.glob(os.path.join(train_image_dir, "*.png")))
if len(train_images) == 0:
    train_images = sorted(glob.glob(os.path.join(train_image_dir, "*.bmp")))

train_masks = sorted(glob.glob(os.path.join(train_mask_dir, "*.png")))
if len(train_masks) == 0:
    train_masks = sorted(glob.glob(os.path.join(train_mask_dir, "*.bmp")))

test_images = sorted(glob.glob(os.path.join(test_image_dir, "*.jpg")))
if len(test_images) == 0:
    test_images = sorted(glob.glob(os.path.join(test_image_dir, "*.png")))
if len(test_images) == 0:
    test_images = sorted(glob.glob(os.path.join(test_image_dir, "*.bmp")))

test_masks = sorted(glob.glob(os.path.join(test_mask_dir, "*.png")))
if len(test_masks) == 0:
    test_masks = sorted(glob.glob(os.path.join(test_mask_dir, "*.bmp")))

if len(test_images) == 0 or test_image_dir == train_image_dir:
    from sklearn.model_selection import train_test_split
    train_images, test_images, train_masks, test_masks = train_test_split(
        train_images, train_masks, test_size=0.2, random_state=42
    )

print(f"Found {len(train_images)} training images and {len(test_images)} test images")

if len(train_images) == 0:
    raise ValueError("No images found! Check the dataset structure and file extensions.")

image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

train_dataset = UnderwaterSegDataset(train_images, train_masks, transform=image_transforms, mask_transform=mask_transforms)
test_dataset = UnderwaterSegDataset(test_images, test_masks, transform=image_transforms, mask_transform=mask_transforms)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)
    img = img * std + mean
    img = np.clip(img, 0, 1)
    return img

for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis('off')
    axes[1].imshow(mask.squeeze(), cmap='tab20')
    axes[1].set_title("Mask")
    axes[1].axis('off')
    plt.show()

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)

In [ ]:
# TO DO
from tqdm import tqdm
import torch.nn as nn

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 5

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()

In [ ]:
# TO DO
model.eval()

with torch.no_grad():
    for i in range(3):
        img, mask = test_dataset[i]
        img_input = img.unsqueeze(0).to(device)

        pred = model(img_input)
        pred_mask = torch.argmax(pred, dim=1).squeeze().cpu()

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(denormalize(img))
        axes[0].set_title("Image")
        axes[0].axis('off')

        axes[1].imshow(mask.squeeze(), cmap='tab20')
        axes[1].set_title("Ground Truth")
        axes[1].axis('off')

        axes[2].imshow(pred_mask, cmap='tab20')
        axes[2].set_title("Prediction")
        axes[2].axis('off')

        plt.show()